> extract pins from processed image  

In [ ]:
#| default_exp model_eval.extract_problem_pins

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import pandas as pd
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from fastcore.all import *
import matplotlib as mpl
import matplotlib.pyplot as plt
from fastcore.all import *
import shutil
import cv2
from typing import Union, List, Tuple, Dict
import pandas as pd
from skimage import io, morphology, measure
from scipy.ndimage import (label, sum, binary_fill_holes)
import os
from tqdm.auto import tqdm
from argparse import ArgumentParser

In [ ]:
#| export
import sys
sys.path.append('/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append('/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')

In [ ]:
#| export
from cv_tools.core import * 

In [ ]:
#| exporti
from private_easy_pin_detection.core import *

In [ ]:
#| export
def extract_prob_frm_pr_img(
        pr_img:np.ndarray, # processed image, only opencv
        img:np.ndarray,
        msk_img:np.ndarray, # mask, in case one needs to get also the mask from the image
        only_quadrilaterals:bool=True, # threshold for area
        save_img_path:Union[str, Path]=None, # save path
        save_im_name:str='cut_img', # save name
        msk_save_path:Union[str, Path]=None, # mask save path
        msk_name:str='mask', # mask name
        show:bool=False
        ):

    'cut specific parts from the processed image, where model failed'
    Path(save_img_path).mkdir(exist_ok=True, parents=True)

    hsv = cv2.cvtColor(pr_img, cv2.COLOR_RGB2HSV)
    # My mistage i am reading bgr image. so change in color space


    # searching for blue color, after reading processed image, it seems problem pins are not red, but blue
    # therefore blue color is searched
    mask = cv2.inRange(hsv, (100,90, 255 ), (120,255, 255))

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for i, contour in enumerate(contours):
        # Approximate contour to polygon
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        # Check if the polygon has 4 sides
        if len(approx) == 4 or not only_quadrilaterals:
            x, y, w, h = cv2.boundingRect(contour)
            if save_img_path is not None:
                cut_img = img[y:y+h, x:x+w]
                if i>0:
                    save_im_name = f'{Path(save_im_name).stem}_{i}'
                    cv2.imwrite(f'{save_img_path}/{save_im_name}.png', cut_img)
                else:
                    cv2.imwrite(f'{save_img_path}/{save_im_name}.png', cut_img)

            if msk_save_path is not None:
                if i>0:
                    msk_name = Path(msk_name).stem
                    cv2.imwrite(f'{msk_save_path}/{msk_name}_{i}.png', msk_img[y:y+h, x:x+w])
                else:
                    cv2.imwrite(f'{msk_save_path}/{msk_name}', msk_img[y:y+h, x:x+w])
        if show:
            show_(pr_img[y:y+h, x:x+w])

In [ ]:
#| export
def extract_prob_frm_pr_img_folder(
        im_path:Union[str, Path], # path to images (normally failed images)
        msk_path:Union[str, Path], # path to masks (normally failed masks)
        pr_im_path:Union[str, Path], # path to processed images (normally al cropped processed images)
        save_im_path:Union[str, Path], # save path
        save_pr_im_path:Union[str, Path], # save path for processed images
        save_msk_path:Union[str, Path], # save path for masks
        ):
    'cut specific parts from the processed image, where model failed'
    Path(save_im_path).mkdir(exist_ok=True, parents=True)
    # getting recipe names
    names_ = get_name_(Path(im_path).ls())

    for i in tqdm(names_, total=len(names_)):
        pr_im_name = i.replace('image2', 'processed_image')
        #print(pr_im_name)
        pr_img = read_img(f'{pr_im_path}/{pr_im_name}', cv=True, gray=False)
        img = read_img(f'{im_path}/{i}', cv=True )
        if msk_path is not None:
            msk = read_img(f'{msk_path}/{i}', cv=True, gray=True)
        else:
            msk = None
        extract_prob_frm_pr_img(
            pr_img=pr_img,
            img=img,
            msk_img=msk,
            save_img_path=save_im_path,
            save_im_name=i,
            msk_save_path=save_msk_path,
            msk_name=i,
        )

In [ ]:
#| hide
#| eval: false
im_path = Path('M:/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1709/missing_1709_unzip/main_im2_cropped_masks/v3/failed/missing/images')
msk_path = Path('M:/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1709/missing_1709_unzip/main_im2_cropped_masks/v3/failed/missing/masks')
pr_im_path = Path('M:/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1709/missing_1709_unzip/main_pr_cropped')
save_im_path = Path('M:/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1709/missing_1709_unzip/main_im2_cropped_masks/v3/failed/missing_pin_sn_images')
save_msk_path = Path('M:/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_1709/missing_1709_unzip/main_im2_cropped_masks/v3/failed/missing_pin_sn_masks')
names = get_name_(im_path.ls())
names_ = get_name_(im_path.ls())

for i in tqdm(names_, total=len(names_)):
    pr_im_name = i.replace('image2', 'processed_image')
    #print(pr_im_name)
    pr_img = read_img(f'{pr_im_path}/{pr_im_name}', cv=True, gray=False)
    img = read_img(f'{im_path}/{i}', cv=True )
    msk = read_img(f'{msk_path}/{i}', cv=True, gray=True)
    extract_prob_frm_pr_img(
        pr_img=pr_img,
        img=img,
        msk_img=msk,
        save_img_path=save_im_path,
        save_im_name=i,
        msk_save_path=None,
        msk_name=i,
    )

In [ ]:
#| export
def parse_args_():

    parser = ArgumentParser(description='extract problem pins from processed images')
    parser.add_argument(
        '--im_path', 
        type=str,
          help='path to processed images (normally failed images)')
    parser.add_argument(
        '--msk_path', 
        type=str,
        default=None,
          help='path to masks (normally failed masks)')
    parser.add_argument(
        '--pr_im_path', 
        type=str,
          help='path to processed images (normally al cropped processed images)')
    parser.add_argument(
        '--save_im_path', 
        type=str,
          help='save path')
    parser.add_argument(
        '--save_msk_path', 
        default=None,
        type=str,
          help='save path for masks')
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()
    extract_prob_frm_pr_img_folder(
        im_path=args.im_path,
        msk_path=args.msk_path,
        pr_im_path=args.pr_im_path,
        save_im_path=args.save_im_path,
        save_pr_im_path=args.save_im_path,
        save_msk_path=args.save_msk_path,
    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('07_extract_problems_from_processed_image.ipynb')